# Part 1.2 Workflows and agents

After this notebook you can tell a workflow from an agent, plan one with the five
"thinking in LangGraph" steps, and pick between LangGraph and Deep Agents.

## See it

![Workflows vs agents](../../assets/workflows-vs-agent-light.svg)

A **workflow** runs predetermined code paths: the code decides what runs next, and the
LLM works inside the steps. An **agent** defines its own process: the model decides
which tool to call and when to stop.

Kea Air's baggage claims is a workflow. You can draw every step in advance:

```
START -> intake -> validate_photo -> assess_damage -> check_history -> apply_policy
      -> [human_review] -> draft_letter -> record_claim -> END
(no photo, or not luggage: -> reject_letter -> END)
```

Part 1 uses two workflow patterns: **prompt chaining** (each LLM step feeds the next)
and **routing** (a function picks the next step). Part 2 is an agent.

## Build it

Plan the claims workflow with the five steps from
[Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph).

| Step | Question | Claims workflow |
| --- | --- | --- |
| 1. Map the steps | What are the discrete steps, and where does the path split? | Check photo, assess damage, look up history, apply policy, optional human review, write letter, record claim. Two splits: not luggage, and needs review. |
| 2. Classify each step | LLM step, data step, action step, or user input step? | **LLM:** `validate_photo`, `assess_damage`, `draft_letter`. **Data:** `check_history`. **Action:** `record_claim`. **User input:** `human_review`. **Plain code:** `apply_policy`, `reject_letter`. |
| 3. Design the state | What must persist between steps? Store raw data, not formatted text. | The form fields, `photo_check`, `assessment`, `prior_claims`, `policy`, `review`, `payout_usd`, `letter`. The prompt text is built inside each node. |
| 4. Build the nodes | Each node reads state and returns updates. How does it fail? | Retry the LLM nodes. Pause with `interrupt()` for a human. Let unexpected errors surface. |
| 5. Wire it together | Which edges are fixed, which are conditional? | Two conditional edges (`route_after_photo_check`, `route_after_policy`). Compile with a checkpointer so the pause can resume. |

The finished graph in `kea/claims/graph.py` follows this plan.

## Under the hood

**LangGraph vs Deep Agents.** The two sit in the same stack, and Deep Agents is built on
LangGraph.

<img src="../../assets/slides/runtime-framework-harness.jpg" width="720">

LangGraph is the **runtime**: nodes, edges, state. LangChain is the **framework**: an
LLM node with tools in a loop. Deep Agents is the **harness**: that loop plus a
filesystem, a planning tool, and subagents.

<img src="../../assets/slides/oss-stack.jpg" width="720">

Each layer adds to the one below. LangGraph gives you checkpointing, durable execution,
and state management. LangChain adds `create_agent`, tool calling, and middleware. Deep
Agents adds a filesystem, memory, subagents, and context management.

| Reach for | When |
| --- | --- |
| **LangGraph** | You need fine control of the path, or can draw the steps in advance. Workflows that mix fixed and LLM steps. Long-running, stateful runs that must survive failure. |
| **Deep Agents** | The task is open-ended and long, and the model should plan, use files, and delegate. You want those tools and context management built in. |

A workflow is cheaper and more predictable, because the same claim takes the same path.
An agent copes with problems you can't enumerate, at the price of less control. Part 1
is the first case, Part 2 the second.

Docs:
[Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents),
[Choosing a product](https://docs.langchain.com/oss/concepts/products).

## Recap

- A workflow fixes the path in code; an agent lets the model choose.
- Plan with five steps: map, classify, design state, build nodes, wire.
- Keep rules in plain code and use the LLM only where the input is fuzzy.
- LangGraph is the runtime under LangChain and Deep Agents.

**Next:** [Part 1.3 The claims workflow in Studio](03_claims_workflow_in_studio.ipynb)